In [14]:
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pandas as pd

# ── Dados ──────────────────────────────────────────────────────────────────────
P_demand_data = [
    1.9317, 1.6090, 1.4079, 1.3281, 1.3834, 1.6413,
    1.9395, 1.7383, 1.8341, 1.8354, 1.9312, 2.3645,
    2.2038, 2.2997, 2.1659, 2.5046, 2.7490, 4.0597,
    4.9924, 5.4257, 5.0491, 4.4294, 3.7692, 2.7716
]

P_pv_data = [
    0.0000, 0.0000, 0.0000, 0.0000, 0.0796, 0.4565,
    1.0742, 1.5790, 2.4343, 2.7488, 3.5092, 3.8988,
    3.9734, 3.7105, 3.1671, 2.7282, 2.3926, 2.1764,
    1.9083, 1.4257, 0.0034, 0.0000, 0.0000, 0.0000
]

tariff_buy = [
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.32629,
    0.51792, 0.51792, 0.51792, 0.32629, 0.22419, 0.22419
]

parameters = {
    "General": {"timestep": 60},
    "Grid":    {"Pmax": 90},
    "BESS": {
        "Pmax": 2.0,
        "eff": 0.90,
        "self_discharge": 0.01,
        "capacity": 7,
        "initial capacity": 0,
    }
}


# ── Definição dos cenários ─────────────────────────────────────────────────────
scenarios = {
    "base": {
        "P_demand": P_demand_data,
        "P_pv":     P_pv_data,
        "prob":     0.34,  
    },
    "alta_demanda": {
        "P_demand": [round(x * 1.3, 4) for x in P_demand_data],
        "P_pv":     [round(x * 0.7, 4) for x in P_pv_data],
        "prob":     0.33,
    },
    "alta_geracao": {
        "P_demand": [round(x * 0.7, 4) for x in P_demand_data],
        "P_pv":     [round(x * 1.3, 4) for x in P_pv_data],
        "prob":     0.33,
    },
}

# ── Classes ────────────────────────────────────────────────────────────────────
class General:
    def __init__(self, parameters):
        self.timestep = parameters["General"]["timestep"]

class BESS:
    def __init__(self, parameters, general):
        self.general          = general
        self.Pmax             = parameters["BESS"]["Pmax"]
        self.eff              = parameters["BESS"]["eff"]
        self.initial_capacity = parameters["BESS"]["initial capacity"]
        self.capacity         = parameters["BESS"]["capacity"]
        self.self_discharge   = parameters["BESS"]["self_discharge"]

class Grid:
    def __init__(self, parameters, general):
        self.general = general
        self.Pmax    = parameters["Grid"]["Pmax"]

class PV:
    def __init__(self, parameters, general):
        self.parameters = parameters
        self.general = general

class SmartHome:
    def __init__(self, parameters, P_demand_data, P_pv_data, tariff_buy):
        self.P_demand_data = P_demand_data
        self.P_pv_data = P_pv_data
        self.tariff_buy = tariff_buy
        self.general = General(parameters)
        self.grid    = Grid(parameters, self.general)
        self.bess    = BESS(parameters, self.general)
        self.pv      = PV(parameters, self.general)

    def build(self):
        model = pyo.ConcreteModel('SmartHome')
        delta = self.general.timestep / 60  # 1.0 hora

        # ── Índices ────────────────────────────────────────────────────────────
        # Conjunto de horas (0 a 23)
        model.T = pyo.RangeSet(0, len(self.P_demand_data) - 1)

        # ── Parâmetros ────────────────────────────────────────────────────────────
        # Demanda de potência
        model.P_demand = pyo.Param(model.T, initialize=self.P_demand_data)

        # Geração solar (PV)
        model.P_pv = pyo.Param(model.T, initialize=self.P_pv_data)

        # Tarifa de eletricidade
        model.tariff = pyo.Param(model.T, initialize=self.tariff_buy)

        # ── Variáveis ──────────────────────────────────────────────────────────────
        # Potência comprada (+) da rede
        model.Pgrid_buy = pyo.Var(model.T,
                              bounds=(0, self.grid.Pmax),
                              within=pyo.NonNegativeReals)

        # Potência vendida à rede (excedente PV)
        model.Pgrid_sell = pyo.Var(model.T,
                                   bounds=(0, self.grid.Pmax),
                                   within=pyo.NonNegativeReals)

        # Potência de carga da bateria (carregando)
        model.Pbess_charge = pyo.Var(model.T,
                                     bounds=(0, self.bess.Pmax),
                                     within=pyo.NonNegativeReals)

        # Potência de descarga da bateria (descarregando)
        model.Pbess_discharge = pyo.Var(model.T,
                                        bounds=(0, self.bess.Pmax),
                                        within=pyo.NonNegativeReals)

        # Energia armazenada na bateria
        model.E_bess = pyo.Var(model.T,
                               bounds=(0, self.bess.capacity),
                               within=pyo.NonNegativeReals)

        # Estado da bateria: 1 = descarregando, 0 = carregando
        model.state = pyo.Var(model.T, within=pyo.Binary)

        model.Pgrid = pyo.Var(model.T, bounds=(-self.grid.Pmax, self.grid.Pmax), within=pyo.Reals)  # Potência líquida da rede (positivo = compra, negativo = venda)

        # ── Restrições ─────────────────────────────────────────────────────────

        # 1) Balanço de potência em cada hora
        #    Pgrid_buy + PV + Pbess_discharge = Demanda + Pbess_charge + Pgrid_sell
        def power_balance_rule(model, t):
            return (+ model.Pgrid[t]
                    + model.P_pv[t]
                    + model.Pbess_discharge[t]
                    ==
                    + model.P_demand[t]
                    + model.Pbess_charge[t]
            )
    
        model.power = pyo.Constraint(model.T, rule=power_balance_rule)

        def grid_balance_rule(model, t):
            return model.Pgrid[t] == model.Pgrid_buy[t] - model.Pgrid_sell[t]
        
        model.grid_balance = pyo.Constraint(model.T, rule=grid_balance_rule)


        # 2) Dinâmica da bateria hora a hora
        #    E[t] = E[t-1] + eff*delta*Pch[t] - delta*Pdis[t]/eff - beta*delta*E[t]
        def bess_energy_rule(model, t):
            eff  = self.bess.eff
            beta = self.bess.self_discharge

            charge_term    = eff  * delta * model.Pbess_charge[t]
            discharge_term = delta * model.Pbess_discharge[t] / eff
            loss_term      = beta * delta * model.E_bess[t]

            if t == 0:
                E_prev = self.bess.initial_capacity  # hora 0: sem hora anterior
            else:
                E_prev = model.E_bess[t - 1]

            return model.E_bess[t] == E_prev + charge_term - discharge_term - loss_term

        model.bess_energy = pyo.Constraint(model.T, rule=bess_energy_rule)

        # 3) Bateria só pode carregar OU descarregar (não os dois ao mesmo tempo)
    
        def bess_discharge_limit(model, t): # (só descarrega se state=1)
            return model.Pbess_discharge[t] <= self.bess.Pmax * model.state[t] 

        def bess_charge_limit(model, t): # (só carrega se state=0)
            return model.Pbess_charge[t] <= self.bess.Pmax * (1 - model.state[t])

        model.bess_dis_limit = pyo.Constraint(model.T, rule=bess_discharge_limit)
        model.bess_ch_limit  = pyo.Constraint(model.T, rule=bess_charge_limit)

        # ── Função objetivo ────────────────────────────────────────────────────
        # Minimizar custo total de compra de energia da rede
        def objective_rule(model):
            return delta * sum( model.tariff[t] * model.Pgrid_buy[t]
                                - (model.tariff[t]*0.7) * model.Pgrid_sell[t]
                                for t in model.T)

        model.funcao_objetivo = pyo.Objective(rule=objective_rule,
                                              sense=pyo.minimize)
        self.model = model

    def solve(self):
        solver   = SolverFactory('highs')
        solution = solver.solve(self.model)

        # Verifica se resolveu com sucesso
        if (solution.solver.status == SolverStatus.ok and
            solution.solver.termination_condition == TerminationCondition.optimal):
            print("\n✓ Solução ótima encontrada!")
            print(f"  Custo total: R$ {pyo.value(self.model.funcao_objetivo):.2f}")
        else:
            print("\n✗ Solução não ótima. Verifique o status do solver:")
            print(f"  Status: {solution.solver.status}")
            print(f"  Termination condition: {solution.solver.termination_condition}")

        # Exibir valores das variáveis
        print("\n" + "="*80)
        print("VALORES DAS VARIÁVEIS")
        
        results = []
        for t in self.model.T:
            results.append({
                'Hora': t,
                'P_Grid_Buy (kW)': f"{pyo.value(self.model.Pgrid_buy[t]):.2f}",
                'P_Grid_Sell (kW)': f"{pyo.value(self.model.Pgrid_sell[t]):.2f}",
                'PV (kW)': f"{pyo.value(self.model.P_pv[t]):.2f}",
                "P demand (kW)": f"{pyo.value(self.model.P_demand[t]):.2f}",
                'P_BESS_Charge (kW)': f"{pyo.value(self.model.Pbess_charge[t]):.2f}",
                'P_BESS_Discharge (kW)': f"{pyo.value(self.model.Pbess_discharge[t]):.2f}",
                'E_BESS (kWh)': f"{pyo.value(self.model.E_bess[t]):.2f}"
            })
        
        results_df = pd.DataFrame(results)
        print(results_df.to_string(index=False))

# ── Execução ───────────────────────────────────────────────────────────────────

# ── Resolver cada cenário e coletar resultados ─────────────────────────────────
all_results = {}

for case, dados in scenarios.items():
    print("\n" + "="*80)
    print(f"  Cenário: {case}  (prob = {dados['prob']})")

    sh = SmartHome(parameters, dados["P_demand"], dados["P_pv"], tariff_buy)
    sh.build()
    sh.solve()

    all_results[case] = sh 

print("\n── Resumo por cenário ────────────────────────────────────")
for case, sh in all_results.items():
    custo = pyo.value(sh.model.funcao_objetivo)
    prob  = scenarios[case]["prob"]
    print(f"  {case:>15}: R$ {custo:.4f}  (probabilidade: {prob})")

custo_esperado = sum(
    pyo.value(sh.model.funcao_objetivo) * scenarios[case]["prob"]
    for case, sh in all_results.items()    
    )
print(f"\n  Custo esperado total (E[C]): R$ {custo_esperado:.4f}")


  Cenário: base  (prob = 0.34)

✓ Solução ótima encontrada!
  Custo total: R$ 8.79

VALORES DAS VARIÁVEIS
 Hora P_Grid_Buy (kW) P_Grid_Sell (kW) PV (kW) P demand (kW) P_BESS_Charge (kW) P_BESS_Discharge (kW) E_BESS (kWh)
    0            1.93             0.00    0.00          1.93               0.00                  0.00         0.00
    1            1.61             0.00    0.00          1.61               0.00                 -0.00         0.00
    2            1.41             0.00    0.00          1.41               0.00                 -0.00         0.00
    3            1.33             0.00    0.00          1.33               0.00                 -0.00         0.00
    4            1.30             0.00    0.08          1.38               0.00                 -0.00         0.00
    5            1.18             0.00    0.46          1.64               0.00                 -0.00         0.00
    6            0.87             0.00    1.07          1.94               0.00         